%md
# Notebook 14: Unity Catalog Governance

**Question:** How do we make the platform enforce data rules automatically, instead of relying on people to remember them?

**Builds on:** Notebook 13's DME tables (bronze contains provider names and addresses; silver and gold do not).

**Governance rule being enforced:** No individual provider names or identifiers in any public output.

In [0]:
TABLE = "workspace.default.dme_referring_2024_bronze"

classification = {
    "Rfrg_NPI": "restricted",
    "Rfrg_Prvdr_Last_Name_Org": "restricted", "Rfrg_Prvdr_First_Name": "restricted", "Rfrg_Prvdr_MI": "restricted",
    "Rfrg_Prvdr_St1": "restricted", "Rfrg_Prvdr_St2": "restricted",
    "Rfrg_Prvdr_City": "internal", "Rfrg_Prvdr_Zip5": "internal",
    "Rfrg_Prvdr_Crdntls": "public", "Rfrg_Prvdr_State_Abrvtn": "public", "Rfrg_Prvdr_Spclty_Desc": "public",
}

for col, tier in classification.items():
    spark.sql(f"ALTER TABLE {TABLE} ALTER COLUMN {col} SET TAGS ('classification' = '{tier}')")

spark.sql(f"COMMENT ON TABLE {TABLE} IS 'Raw CMS DME 2024. Contains provider names and addresses (restricted).'")

display(spark.sql("""
    SELECT column_name, tag_value AS classification
    FROM workspace.information_schema.column_tags
    WHERE table_name = 'dme_referring_2024_bronze' AND tag_name = 'classification'
    ORDER BY tag_value, column_name"""))

In [0]:
TABLE = "workspace.default.dme_referring_2024_bronze"

spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.mask_restricted_str(val STRING)
RETURNS STRING
RETURN CASE WHEN is_account_group_member('phi_reviewers') THEN val ELSE '***RESTRICTED***' END
""")

for col in ["Rfrg_Prvdr_Last_Name_Org", "Rfrg_Prvdr_First_Name", "Rfrg_Prvdr_MI",
            "Rfrg_Prvdr_St1", "Rfrg_Prvdr_St2"]:
    spark.sql(f"ALTER TABLE {TABLE} ALTER COLUMN {col} SET MASK workspace.default.mask_restricted_str")

distinct_names = spark.sql(f"SELECT COUNT(DISTINCT Rfrg_Prvdr_Last_Name_Org) FROM {TABLE}").collect()[0][0]
print(f"Distinct last names visible to me: {distinct_names:,}")
print("Mask is working" if distinct_names == 1 else "WARNING: mask not applied, names are visible")

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW workspace.default.cpap_resupply_share_v
COMMENT 'Shareable CPAP resupply summary. No provider identifiers; groups under 11 prescribers suppressed.'
AS
SELECT HCPCS_CD,
       Rfrg_Prvdr_Spclty_Desc AS specialty,
       COUNT(*) AS prescribers,
       ROUND(percentile_approx(srv_per_patient, 0.5), 2) AS median_per_patient,
       SUM(CASE WHEN robust_z > 3.5  AND Tot_Suplr_Benes >= 20 THEN 1 ELSE 0 END) AS robust_high_flags,
       SUM(CASE WHEN robust_z < -3.5 AND Tot_Suplr_Benes >= 20 THEN 1 ELSE 0 END) AS robust_low_flags
FROM workspace.default.dme_cpap_peer_scores_gold
GROUP BY HCPCS_CD, Rfrg_Prvdr_Spclty_Desc
HAVING COUNT(*) >= 11
""")

v = spark.table("workspace.default.cpap_resupply_share_v")
print("Columns:", v.columns)
assert not any("NPI" in c for c in v.columns), "Identifier column found in shareable view"
print(f"Rows: {v.count():,} | Identifier check passed")
display(v.orderBy("HCPCS_CD", "specialty").limit(10))

In [0]:
display(spark.sql("DESCRIBE HISTORY workspace.default.dme_referring_2024_bronze")
        .select("version", "timestamp", "operation", "operationParameters"))

## Findings: Notebook 14

**Why govern public data:** CMS publishes this data openly. The risk comes from what we add to it. An outlier flag attached to a name reads like an accusation, with no context. Governance keeps our derived findings from harming a physician or supplier unfairly.

**Classification (Unity Catalog tags on 11 columns):** Restricted: NPI, names, street address. Internal: city, ZIP (quasi-identifiers; combinations can identify a person). Public: state, specialty, credentials.

**Enforced by the platform:**
- Names and street addresses are masked for anyone outside an approved group, including the table owner. Verified without displaying a single name (distinct count = 1).
- The shareable view `cpap_resupply_share_v` has 6 columns and no identifiers. Groups with fewer than 11 prescribers are suppressed, and an `assert` stops the notebook if an identifier column ever appears. 157 groups.
- Lineage is recorded automatically, and shows the NPI stops at the gold table and never reaches the shareable view.

**Defense in depth:** The 11-prescriber rule removed nothing, because Notebook 13 already required 30+ per peer group. The rule stays so the view doesn't depend on upstream settings.

**Accepted risk:** The NPI stays unmasked in bronze because the Notebook 13 pipeline groups by it. Contained by never displaying it and excluding it from shared views.

**Also observed:** CPAP resupply orders spread well beyond sleep practices. For heated tubing alone: internal medicine 2,945 prescribers, family practice 2,896, nurse practitioners 2,842.

**What I'd tell a COO:** Public data still needs governance, because our analysis creates new information about real people. Names and addresses are now protected by the platform, not by habit, and anything we share comes from a view that can't expose identifiers.

**What this workspace can't do:** Tag and mask changes don't appear in the table's history, so some governance changes have no visible audit trail here. The NPI is unmasked. Access control is tested with one user, not a real team.

**What I'd need in production:** The platform audit log for governance changes; groups and role-based access; pipelines running under an authorized service account so the NPI can be masked for people; and periodic access reviews.